<a href="https://colab.research.google.com/github/bahaali273/COMP9170-Hospital-Readmission/blob/bahaa-dev/notebooks/03_kumo_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# NVIDIA Kumo-Tabular — Hospital Readmission Prediction

**Course:** COMP 9170 — Data Science  
**Author:** Bahaa Al-Juboori

## 1. Environment and Compatibility Check

### Decisions
- Investigate NVIDIA Kumo-Tabular for binary classification.
- Check the available GPU and software environment.
- Reuse the team's existing training and test partitions.
- Investigate compatibility with 133 input features.
- Keep the original test set reserved for final evaluation.


### Results
- Python version: 3.13.16.
- PyTorch version: 2.14.1+cu130.
- CUDA available: True.
- GPU: NVIDIA Tesla T4.
- GPU memory: 14.56 GB.
- The GPU environment was configured successfully.
- Kumo-Tabular inference has not yet been tested.



In [1]:

import sys
import torch

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU detected.")


Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56



## 2. NVIDIA Kumo-Tabular Installation

### Decisions
- Use NVIDIA's official structured-data-models package.
- Install the library in the GPU-enabled Colab environment.
- Verify that Kumo-Tabular can be imported successfully.
- Do not load model weights or run predictions yet.


### Results
- NVIDIA structured-data-models version: 0.2.1.
- The SDM package was installed successfully.
- The `sdm` library imported without errors.
- The KumoTabular class was detected successfully.
- Model weights have not yet been downloaded or loaded.



In [2]:

%pip install -q structured-data-models

import importlib.metadata

print("Installed SDM version:",
      importlib.metadata.version("structured-data-models"))

import sdm

print("SDM import successful!")
print("Kumo-Tabular available:",
      hasattr(sdm.models, "KumoTabular"))


Installed SDM version: 0.2.1
SDM import successful!
Kumo-Tabular available: True



## 3. Dataset Compatibility Check

### Decisions
- Reuse the processed training dataset from the shared pipeline.
- Verify that all input features are numeric and contain no missing values.
- Inspect the binary target and feature count.
- Avoid using the held-out test set during initial compatibility testing.
- Start with a small validation experiment before full-scale inference.

### Results

### Results
- Training dataset: 55,976 rows and 134 columns.
- Number of input features: 133.
- Target classes: 0 and 1.
- All input features are numeric.
- No missing values were detected.
- Positive-class prevalence: 8.97%.
- Dataset memory usage: 57.23 MB.
- The dataset passed initial compatibility checks for Kumo-Tabular.
- Model inference and GPU memory compatibility have not yet been tested.



In [4]:

import pandas as pd
import numpy as np

train_df = pd.read_csv("/content/train_clean.csv")

target = "readmit_30"
X = train_df.drop(columns=[target])
y = train_df[target]

print("Training shape:", train_df.shape)
print("Number of features:", X.shape[1])
print("Target classes:", sorted(y.unique().tolist()))
print("Non-numeric features:", X.select_dtypes(exclude="number").columns.tolist())
print("Missing values:", int(train_df.isna().sum().sum()))
print("Target prevalence:", round(y.mean(), 4))
print("Memory usage (MB):",
      round(train_df.memory_usage(deep=True).sum() / 1024**2, 2))


Training shape: (55976, 134)
Number of features: 133
Target classes: [0, 1]
Non-numeric features: []
Missing values: 0
Target prevalence: 0.0897
Memory usage (MB): 57.23



## 4. Small In-Context Dataset Preparation

### Decisions
- Use the existing cleaned training dataset.
- Create a stratified development subset with 500 context rows and 100 query rows.
- Keep the held-out test dataset untouched.
- Convert the data to NVIDIA's TableTensor format.
- Mark the binary target as categorical.
- Verify successful conversion before loading model weights.


### Results
- Selected 600 samples from the processed training dataset.
- Context dataset: 500 samples.
- Query dataset: 100 samples.
- Input features: 133.
- Context positive-class prevalence: 9.0%.
- Query positive-class prevalence: 9.0%.
- Device: CUDA GPU.
- Successfully inferred column types using `sdm.infer_stypes()`.
- Successfully converted the dataset to NVIDIA TableTensor format.
- No model inference has been performed yet.
- The held-out test dataset remains untouched.



In [5]:

import torch
import sdm
from sklearn.model_selection import train_test_split

SEED = 42
TARGET = "readmit_30"

# Create a stratified development sample
dev_df, _ = train_test_split(
    train_df,
    train_size=600,
    stratify=train_df[TARGET],
    random_state=SEED
)

# Separate context rows from query rows
context_df, query_df = train_test_split(
    dev_df,
    train_size=500,
    stratify=dev_df[TARGET],
    random_state=SEED
)

# Recombine for consistent TableTensor conversion
sample_df = pd.concat(
    [context_df, query_df],
    ignore_index=True
)

# NVIDIA Kumo-Tabular input format
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

stypes = sdm.infer_stypes(
    sample_df,
    overrides={TARGET: "categorical"}
)

table = sdm.TableTensor.from_pandas(
    df=sample_df,
    stypes=stypes,
    device=device
)

print("Context rows:", len(context_df))
print("Query rows:", len(query_df))
print("Number of features:", sample_df.shape[1] - 1)
print("Context positive rate:",
      round(context_df[TARGET].mean(), 4))
print("Query positive rate:",
      round(query_df[TARGET].mean(), 4))
print("Device:", device)
print("TableTensor conversion successful!")


Context rows: 500
Query rows: 100
Number of features: 133
Context positive rate: 0.09
Query positive rate: 0.09
Device: cuda
TableTensor conversion successful!



## 5. Pretrained Kumo-Tabular Model Loading

### Decisions
- Use NVIDIA's pretrained Kumo-Tabular model.
- Select the small model to reduce GPU memory requirements.
- Configure the model for binary classification.
- Run inference on the Tesla T4 GPU.
- Verify successful initialization before prediction.
- Keep the held-out test set untouched.


### Results
- Successfully downloaded and initialized the pretrained Kumo-Tabular Small model.
- Model task: Classification.
- Model weights downloaded: approximately 110 MB.
- Inference device: NVIDIA Tesla T4 (CUDA).
- GPU allocated memory after loading: 0.11 GB.
- GPU reserved memory after loading: 0.12 GB.
- Hugging Face displayed an unauthenticated-request warning, but model loading succeeded.
- Model inference has not yet been performed.
- The held-out test set remains untouched.



In [6]:

import torch
import sdm

# Use the available GPU
device = torch.device("cuda")

# Load pretrained Kumo-Tabular
kumo_model = sdm.models.KumoTabular(
    task="classification",
    size="small",
    device=device
)

print("Kumo-Tabular loaded successfully!")
print("Model size: small")
print("Task: classification")
print("Device:", device)

# Check GPU memory usage
print(
    "GPU allocated memory (GB):",
    round(torch.cuda.memory_allocated() / 1024**3, 2)
)

print(
    "GPU reserved memory (GB):",
    round(torch.cuda.memory_reserved() / 1024**3, 2)
)


config.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

small/classifier.pt: reconstructing file:   0%|          |  0.00B /  110MB            

small/classifier.pt: downloading bytes:           |  0.00B            

Kumo-Tabular loaded successfully!
Model size: small
Task: classification
Device: cuda
GPU allocated memory (GB): 0.11
GPU reserved memory (GB): 0.12



## 6. First In-Context Prediction

### Decisions
- Use the pretrained Kumo-Tabular Small classifier.
- Provide 500 labeled context rows.
- Predict 100 query rows from the training-derived development sample.
- Exclude query target labels from model inputs.
- Use FP16 mixed precision for GPU inference.
- Start with one estimator to test feasibility.
- Keep the held-out test set untouched.

### Results
- Successfully completed the first Kumo-Tabular inference.
- Model: NVIDIA Kumo-Tabular Small.
- Context samples: 500 labeled records.
- Query samples: 100 unlabeled records.
- Prediction output: TableTensor with shape (100, 2).
- Inference time: 6.70 seconds.
- Peak GPU allocated memory: 0.34 GB.
- Inference device: NVIDIA Tesla T4.
- No model fine-tuning was performed.
- Predictive performance has not yet been evaluated.
- The held-out test set remains untouched.



In [7]:

import torch
import time

# Context features and labels
x_context = table[:500].drop_columns(TARGET)
y_context = table[:500, TARGET]

# Query features only: do not expose the target
x_query = table[500:].drop_columns(TARGET)

print("Running first Kumo-Tabular prediction...")

start = time.perf_counter()

kumo_model.eval()

with torch.inference_mode():
    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        probs = kumo_model(
            x_context=x_context,
            y_context=y_context,
            x_query=x_query,
            num_estimators=1
        )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print("Inference completed!")
print("Inference time (seconds):", round(elapsed, 2))
print("Prediction output:")
print(probs)
print("Peak allocated GPU memory (GB):",
      round(torch.cuda.max_memory_allocated() / 1024**3, 2))


Running first Kumo-Tabular prediction...
Inference completed!
Inference time (seconds): 6.7
Prediction output:
TableTensor(
  size=(100, 2),
  blocks={
    numerical (2): [0, 1],
  },
  device=cuda:0,
)
Peak allocated GPU memory (GB): 0.34



## 7. Initial Kumo-Tabular Evaluation

### Decisions
- Evaluate the 100 query predictions from the development sample.
- Use only query labels that were excluded from model inputs.
- Calculate Average Precision (PR-AUC) and ROC-AUC.
- Calculate Precision, Recall, and F1 at threshold 0.5.
- Treat results as a feasibility test, not final model performance.
- Keep the held-out test set untouched.


### Results
- Evaluated 100 query samples from the training-derived development subset.
- Positive-class samples: 9 (9.0%).
- PR-AUC (Average Precision): 0.0936.
- ROC-AUC: 0.4945.
- Precision at threshold 0.5: 0.0000.
- Recall at threshold 0.5: 0.0000.
- F1-score at threshold 0.5: 0.0000.
- No positive cases were correctly identified at the default threshold.
- The PR-AUC was close to the positive-class prevalence (0.09).
- The ROC-AUC was approximately equivalent to random ranking.
- These results are preliminary because the evaluation sample contains only nine positive cases.
- A larger development evaluation is required before drawing conclusions about model performance.
- The held-out test set remains untouched.



In [8]:

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# Convert Kumo predictions to a Pandas DataFrame
probs_df = probs.to_pandas()

print("Probability columns:", probs_df.columns.tolist())
print(probs_df.head())

# Identify the positive-class probability column
positive_col = next(
    col for col in probs_df.columns
    if str(col) == "1"
)

y_probs = probs_df[positive_col].to_numpy()
y_true = query_df[TARGET].to_numpy()

# Apply a preliminary classification threshold
y_pred = (y_probs >= 0.5).astype(int)

print("\nInitial Kumo Evaluation:")
print("PR-AUC:", round(average_precision_score(y_true, y_probs), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_probs), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("Positive cases:", int(y_true.sum()))


Probability columns: ['0', '1']
          0         1
0  0.911456  0.088544
1  0.913677  0.086323
2  0.968381  0.031619
3  0.930584  0.069415
4  0.981257  0.018743

Initial Kumo Evaluation:
PR-AUC: 0.0936
ROC-AUC: 0.4945
Precision: 0.0
Recall: 0.0
F1: 0.0
Positive cases: 9



## 8. Prediction Probability Analysis

### Decisions
- Inspect the distribution of predicted readmission probabilities.
- Compare predicted probabilities for actual positive and negative patients.
- Count how many predictions exceed different probability thresholds.
- Do not optimize thresholds using the held-out test set.


### Results
- Analyzed predicted readmission probabilities for 100 query patients.
- Mean predicted probability: 0.0972.
- Median predicted probability: 0.0713.
- Minimum probability: 0.0148.
- Maximum probability: 0.4760.
- Actual readmission prevalence: 9.0%.
- Mean probability for negative patients: 0.0988.
- Mean probability for positive patients: 0.0815.
- No predicted probability exceeded the default threshold of 0.50.
- Predicted positive counts:
  - Threshold 0.05: 68 patients.
  - Threshold 0.10: 36 patients.
  - Threshold 0.15: 18 patients.
  - Threshold 0.20: 11 patients.
  - Threshold 0.30: 1 patient.
  - Threshold 0.50: 0 patients.
- The default threshold explains why initial precision, recall, and F1 were zero.
- The model did not demonstrate useful discrimination in this small development sample.
- Further evaluation on a larger development sample is necessary.



In [9]:

import pandas as pd
import numpy as np

analysis_df = pd.DataFrame({
    "Actual": y_true,
    "Readmission_Probability": y_probs
})

print("Probability Summary:")
print(analysis_df["Readmission_Probability"].describe())

print("\nProbabilities by Actual Class:")
print(
    analysis_df.groupby("Actual")[
        "Readmission_Probability"
    ].agg(["count", "mean", "median", "min", "max"])
)

print("\nPredicted Positive Counts by Threshold:")

for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    predictions = (y_probs >= threshold).astype(int)

    print(
        f"Threshold {threshold:.2f}: "
        f"{predictions.sum()} predicted positive"
    )


Probability Summary:
count    100.000000
mean       0.097249
std        0.080521
min        0.014785
25%        0.040156
50%        0.071328
75%        0.121819
max        0.475971
Name: Readmission_Probability, dtype: float64

Probabilities by Actual Class:
        count      mean    median       min       max
Actual                                               
0          91  0.098802  0.069954  0.014785  0.475971
1           9  0.081547  0.091707  0.027638  0.164049

Predicted Positive Counts by Threshold:
Threshold 0.05: 68 predicted positive
Threshold 0.10: 36 predicted positive
Threshold 0.15: 18 predicted positive
Threshold 0.20: 11 predicted positive
Threshold 0.30: 1 predicted positive
Threshold 0.50: 0 predicted positive



## 9. Threshold Sensitivity Analysis

### Decisions
- Evaluate multiple classification thresholds on the development sample.
- Compare precision, recall, F1-score, and predicted positive counts.
- Investigate the effect of class imbalance.
- Do not select a final threshold from this small sample.
- Keep the held-out test set untouched.


### Results
- Evaluated six classification thresholds: 0.05, 0.10, 0.15, 0.20, 0.30, and 0.50.
- The highest observed F1-score was 0.1818 at threshold 0.05.
- At threshold 0.05:
  - Precision: 0.1029.
  - Recall: 0.7778.
  - F1-score: 0.1818.
  - True positives: 7.
  - False positives: 61.
  - False negatives: 2.
  - True negatives: 30.
- Lowering the threshold increased recall but produced many false positives.
- At threshold 0.50, no patients were predicted positive.
- These observations are exploratory because the sample contains only nine positive cases.
- No final classification threshold was selected.
- A larger development evaluation is needed before comparing Kumo-Tabular with the MLP baseline.
- The held-out test set remains untouched.



In [10]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd
import numpy as np

threshold_results = []

for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:

    pred = (y_probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_true, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, pred, zero_division=0
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(threshold_results)

print("Kumo-Tabular Threshold Sensitivity:")
display(threshold_df.round(4))


Kumo-Tabular Threshold Sensitivity:


,Threshold,Precision,Recall,F1,TP,FP,FN,TN
0,0.05,0.1029,0.7778,0.1818,7,61,2,30
1,0.10,0.0556,0.2222,0.0889,2,34,7,57
2,0.15,0.0556,0.1111,0.0741,1,17,8,74
3,0.20,0.0000,0.0000,0.0000,0,11,9,80
4,0.30,0.0000,0.0000,0.0000,0,1,9,90
5,0.50,0.0000,0.0000,0.0000,0,0,9,91



## 10. Larger Development Experiment

### Decisions
- Increase the in-context sample size.
- Use 2,000 labeled context rows and 500 query rows.
- Sample exclusively from the training dataset.
- Preserve the original positive-class proportion using stratified sampling.
- Keep context and query patients separate.
- Keep the held-out test dataset untouched.
- Verify the new sample before running inference.

### Results
- Successfully prepared a larger development sample of 2,500 patient records.
- Context dataset: 2,000 labeled records.
- Query dataset: 500 records.
- Number of input features: 133.
- Positive cases in context: 179 (8.95%).
- Positive cases in query: 45 (9.00%).
- Stratified sampling preserved the original class distribution.
- Context and query subsets contain no overlapping rows.
- Successfully converted the larger sample to NVIDIA TableTensor format.
- Model inference has not yet been performed on this sample.
- The held-out test dataset remains untouched.



In [11]:

from sklearn.model_selection import train_test_split
import pandas as pd
import torch
import sdm

SEED = 42

# Draw a larger development sample
large_dev_df, _ = train_test_split(
    train_df,
    train_size=2500,
    stratify=train_df[TARGET],
    random_state=SEED
)

# Create non-overlapping context and query sets
large_context_df, large_query_df = train_test_split(
    large_dev_df,
    train_size=2000,
    stratify=large_dev_df[TARGET],
    random_state=SEED
)

# Combine for TableTensor conversion
large_sample_df = pd.concat(
    [large_context_df, large_query_df],
    ignore_index=True
)

large_stypes = sdm.infer_stypes(
    large_sample_df,
    overrides={TARGET: "categorical"}
)

large_table = sdm.TableTensor.from_pandas(
    df=large_sample_df,
    stypes=large_stypes,
    device=device
)

print("Context rows:", len(large_context_df))
print("Query rows:", len(large_query_df))
print("Features:", large_sample_df.shape[1] - 1)
print("Context positive cases:",
      int(large_context_df[TARGET].sum()))
print("Query positive cases:",
      int(large_query_df[TARGET].sum()))
print("Context positive rate:",
      round(large_context_df[TARGET].mean(), 4))
print("Query positive rate:",
      round(large_query_df[TARGET].mean(), 4))
print("TableTensor conversion successful!")


Context rows: 2000
Query rows: 500
Features: 133
Context positive cases: 179
Query positive cases: 45
Context positive rate: 0.0895
Query positive rate: 0.09
TableTensor conversion successful!



## 11. Larger Development Inference and Evaluation

### Decisions
- Use the existing pretrained Kumo-Tabular Small model.
- Provide 2,000 labeled context rows.
- Predict readmission probabilities for 500 query rows.
- Exclude query labels from model inputs.
- Use FP16 mixed precision for GPU inference.
- Measure inference time and GPU memory usage.
- Evaluate PR-AUC, ROC-AUC, Precision, Recall, and F1.
- Keep the held-out test set untouched.


### Results
- Successfully completed a larger Kumo-Tabular inference experiment.
- Model: NVIDIA Kumo-Tabular Small.
- Context samples: 2,000 labeled patient records.
- Query samples: 500 patient records.
- Positive query cases: 45 (9.0%).
- PR-AUC (Average Precision): 0.2192.
- ROC-AUC: 0.6170.
- Precision at threshold 0.5: 0.0000.
- Recall at threshold 0.5: 0.0000.
- F1-score at threshold 0.5: 0.0000.
- Predicted positive cases at threshold 0.5: 0.
- Inference time: 0.41 seconds.
- Peak GPU allocated memory: 0.73 GB.
- Performance improved compared with the initial small-sample experiment, but the experiments used different samples.
- Runtime comparisons may be influenced by GPU warm-up and caching.
- No final threshold has been selected.
- The held-out test dataset remains untouched.



In [12]:

import torch
import time
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# Separate context and query inputs
large_x_context = large_table[:2000].drop_columns(TARGET)
large_y_context = large_table[:2000, TARGET]
large_x_query = large_table[2000:].drop_columns(TARGET)

# Reset GPU peak memory statistics
torch.cuda.reset_peak_memory_stats()

print("Running larger Kumo-Tabular experiment...")

start = time.perf_counter()

kumo_model.eval()

with torch.inference_mode():
    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        large_probs = kumo_model(
            x_context=large_x_context,
            y_context=large_y_context,
            x_query=large_x_query,
            num_estimators=1
        )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

# Extract positive-class probabilities
large_probs_df = large_probs.to_pandas()
large_y_probs = large_probs_df["1"].to_numpy()
large_y_true = large_query_df[TARGET].to_numpy()

# Preliminary threshold
large_y_pred = (large_y_probs >= 0.5).astype(int)

print("\nLarger Kumo-Tabular Results:")
print("PR-AUC:",
      round(average_precision_score(
          large_y_true, large_y_probs
      ), 4))
print("ROC-AUC:",
      round(roc_auc_score(
          large_y_true, large_y_probs
      ), 4))
print("Precision:",
      round(precision_score(
          large_y_true, large_y_pred,
          zero_division=0
      ), 4))
print("Recall:",
      round(recall_score(
          large_y_true, large_y_pred,
          zero_division=0
      ), 4))
print("F1:",
      round(f1_score(
          large_y_true, large_y_pred,
          zero_division=0
      ), 4))

print("\nInference time (seconds):",
      round(elapsed, 2))
print("Peak GPU memory (GB):",
      round(torch.cuda.max_memory_allocated() / 1024**3, 2))

print("Positive cases:", int(large_y_true.sum()))
print("Predicted positive cases:", int(large_y_pred.sum()))


Running larger Kumo-Tabular experiment...

Larger Kumo-Tabular Results:
PR-AUC: 0.2192
ROC-AUC: 0.617
Precision: 0.0
Recall: 0.0
F1: 0.0

Inference time (seconds): 0.41
Peak GPU memory (GB): 0.73
Positive cases: 45
Predicted positive cases: 0



## 12. Larger Development Threshold Analysis

### Decisions
- Evaluate classification thresholds on the 500-query development sample.
- Measure precision, recall, F1-score, and confusion matrix counts.
- Compare threshold sensitivity with the smaller experiment.
- Do not use the held-out test dataset.
- Treat the analysis as exploratory rather than final threshold selection.


### Results
- Evaluated eight thresholds from 0.05 to 0.50 on 500 development query samples.
- The highest observed F1-score among tested thresholds was 0.2424 at threshold 0.15.
- At threshold 0.15:
  - Precision: 0.3810.
  - Recall: 0.1778.
  - F1-score: 0.2424.
  - True positives: 8.
  - False positives: 13.
  - False negatives: 37.
  - True negatives: 442.
- At threshold 0.10, recall increased to 0.4889, but precision decreased to 0.1384.
- At threshold 0.05, recall was 1.0000, but all 500 query patients were classified as positive.
- Thresholds from 0.20 to 0.30 identified only one positive patient.
- At threshold 0.50, no positive predictions were produced.
- Threshold 0.15 produced the highest F1 among the tested values, but missed most actual readmissions.
- Threshold selection remains exploratory and has not been finalized.
- The held-out test dataset remains untouched.



In [13]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd
import numpy as np

large_threshold_results = []

for threshold in [
    0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.40, 0.50
]:
    pred = (large_y_probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        large_y_true, pred, labels=[0, 1]
    ).ravel()

    large_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            large_y_true, pred, zero_division=0
        ),
        "Recall": recall_score(
            large_y_true, pred, zero_division=0
        ),
        "F1": f1_score(
            large_y_true, pred, zero_division=0
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

large_threshold_df = pd.DataFrame(
    large_threshold_results
)

display(large_threshold_df.round(4))


,Threshold,Precision,Recall,F1,TP,FP,FN,TN
0,0.05,0.0900,1.0000,0.1651,45,455,0,0
1,0.10,0.1384,0.4889,0.2157,22,137,23,318
2,0.15,0.3810,0.1778,0.2424,8,13,37,442
3,0.20,1.0000,0.0222,0.0435,1,0,44,455
4,0.25,1.0000,0.0222,0.0435,1,0,44,455
5,0.30,1.0000,0.0222,0.0435,1,0,44,455
6,0.40,0.0000,0.0000,0.0000,0,0,45,455
7,0.50,0.0000,0.0000,0.0000,0,0,45,455



## 13. Independent Development Validation Preparation

### Decisions
- Reserve the previous 2,500 development records from further evaluation.
- Sample 500 new query records from the remaining training data.
- Use the previous 2,000 context records without changing them.
- Verify that the new query records do not overlap with previous development records.
- Keep the held-out test dataset untouched.
- Evaluate the previously observed threshold of 0.15 without retuning it on this new sample.


### Results
- Prepared an independent development-validation query sample.
- Reused the existing 2,000 labeled context records.
- Selected 500 new validation records from previously unused training rows.
- New validation positive cases: 45 (9.0%).
- Verified zero row overlap with the previous 2,500-record development sample.
- Successfully converted the context and validation data to TableTensor format.
- The previously observed threshold of 0.15 will be evaluated without retuning.
- The original held-out test dataset remains untouched.


In [14]:

from sklearn.model_selection import train_test_split
import pandas as pd
import sdm

# Identify unused training rows by their original indices
remaining_df = train_df.drop(index=large_dev_df.index)

# Sample new independent validation queries
new_query_df, _ = train_test_split(
    remaining_df,
    train_size=500,
    stratify=remaining_df[TARGET],
    random_state=123
)

# Combine existing context with new validation queries
validation_sample_df = pd.concat(
    [large_context_df, new_query_df],
    ignore_index=True
)

validation_stypes = sdm.infer_stypes(
    validation_sample_df,
    overrides={TARGET: "categorical"}
)

validation_table = sdm.TableTensor.from_pandas(
    df=validation_sample_df,
    stypes=validation_stypes,
    device=device
)

overlap = len(
    set(new_query_df.index) &
    set(large_dev_df.index)
)

print("Context rows:", len(large_context_df))
print("New validation rows:", len(new_query_df))
print("Positive validation cases:",
      int(new_query_df[TARGET].sum()))
print("Overlap with previous development sample:", overlap)
print("TableTensor conversion successful!")


Context rows: 2000
New validation rows: 500
Positive validation cases: 45
Overlap with previous development sample: 0
TableTensor conversion successful!



## 14. Independent Development Validation

### Decisions
- Use the pretrained Kumo-Tabular Small model.
- Reuse the existing 2,000 labeled context records.
- Predict outcomes for 500 independent development-validation records.
- Keep validation labels hidden from the model.
- Evaluate the previously selected threshold of 0.15.
- Do not change the threshold based on these validation results.
- Calculate PR-AUC, ROC-AUC, Precision, Recall, F1-score, and confusion matrix.
- Keep the final held-out test dataset untouched.


### Results
- Evaluated Kumo-Tabular Small on 500 independent development-validation patients.
- Reused 2,000 labeled context records.
- Positive validation cases: 45 (9.0%).
- Applied the previously selected threshold of 0.15 without retuning.
- PR-AUC (Average Precision): 0.1016.
- ROC-AUC: 0.5202.
- Precision: 0.0769.
- Recall: 0.0889.
- F1-score: 0.0825.
- True positives: 4.
- False positives: 48.
- False negatives: 41.
- True negatives: 407.
- Inference time: 0.34 seconds.
- Peak GPU allocated memory: 0.73 GB.
- Performance decreased substantially compared with the previous development experiment.
- The validation PR-AUC was close to the 9.0% positive-class prevalence.
- The results suggest limited discrimination and sensitivity to the evaluated sample under the current configuration.
- No additional threshold tuning was performed on the independent validation sample.
- The original held-out test dataset remains untouched.



In [15]:

import torch
import time
import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Separate context and query features
val_x_context = validation_table[:2000].drop_columns(TARGET)
val_y_context = validation_table[:2000, TARGET]
val_x_query = validation_table[2000:].drop_columns(TARGET)

# Fixed threshold from previous development experiment
FIXED_THRESHOLD = 0.15

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.perf_counter()

kumo_model.eval()

with torch.inference_mode():
    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        val_probs = kumo_model(
            x_context=val_x_context,
            y_context=val_y_context,
            x_query=val_x_query,
            num_estimators=1
        )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

# Extract predictions
val_probs_df = val_probs.to_pandas()
val_y_probs = val_probs_df["1"].to_numpy()

# Ground-truth labels used only for evaluation
val_y_true = new_query_df[TARGET].to_numpy()

# Apply fixed threshold
val_y_pred = (
    val_y_probs >= FIXED_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    val_y_true,
    val_y_pred,
    labels=[0, 1]
).ravel()

print("Independent Development Validation Results:")
print("PR-AUC:",
      round(average_precision_score(val_y_true, val_y_probs), 4))
print("ROC-AUC:",
      round(roc_auc_score(val_y_true, val_y_probs), 4))
print("Precision:",
      round(precision_score(val_y_true, val_y_pred, zero_division=0), 4))
print("Recall:",
      round(recall_score(val_y_true, val_y_pred, zero_division=0), 4))
print("F1:",
      round(f1_score(val_y_true, val_y_pred, zero_division=0), 4))

print("\nFixed threshold:", FIXED_THRESHOLD)
print("TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)
print("Inference time (seconds):", round(elapsed, 2))
print("Peak GPU memory (GB):",
      round(torch.cuda.max_memory_allocated() / 1024**3, 2))


Independent Development Validation Results:
PR-AUC: 0.1016
ROC-AUC: 0.5202
Precision: 0.0769
Recall: 0.0889
F1: 0.0825

Fixed threshold: 0.15
TP: 4 FP: 48 FN: 41 TN: 407
Inference time (seconds): 0.34
Peak GPU memory (GB): 0.73



## 15. Kumo-Tabular Experiment Summary

### Decisions
- Consolidate all completed development experiments.
- Distinguish exploratory results from independent validation results.
- Report PR-AUC, ROC-AUC, F1, and evaluation sample sizes.
- Avoid claiming that preliminary improvements generalize.
- Keep the final held-out test set unused.


### Results
- Consolidated three completed Kumo-Tabular development experiments.
- Initial development: 500 context rows and 100 query rows.
- Larger development: 2,000 context rows and 500 query rows.
- Independent validation: 2,000 context rows and 500 new query rows.
- Initial development PR-AUC: 0.0936; ROC-AUC: 0.4945.
- Larger development PR-AUC: 0.2192; ROC-AUC: 0.6170.
- Independent validation PR-AUC: 0.1016; ROC-AUC: 0.5202.
- Independent validation F1-score at the previously selected threshold of 0.15: 0.0825.
- Results indicate substantial variation across development samples.
- The current Kumo-Tabular configuration did not demonstrate consistently strong predictive performance.
- Saved the experiment summary as `kumo_development_summary.csv`.
- The held-out test dataset remains untouched.



In [16]:

import pandas as pd

kumo_summary = pd.DataFrame([
    {
        "Experiment": "Initial development",
        "Context": 500,
        "Query": 100,
        "PR-AUC": 0.0936,
        "ROC-AUC": 0.4945,
        "Threshold": 0.50,
        "F1": 0.0000
    },
    {
        "Experiment": "Larger development",
        "Context": 2000,
        "Query": 500,
        "PR-AUC": 0.2192,
        "ROC-AUC": 0.6170,
        "Threshold": 0.15,
        "F1": 0.2424
    },
    {
        "Experiment": "Independent validation",
        "Context": 2000,
        "Query": 500,
        "PR-AUC": 0.1016,
        "ROC-AUC": 0.5202,
        "Threshold": 0.15,
        "F1": 0.0825
    }
])

display(kumo_summary)

kumo_summary.to_csv(
    "kumo_development_summary.csv",
    index=False
)

print("Saved: kumo_development_summary.csv")


,Experiment,Context,Query,PR-AUC,ROC-AUC,Threshold,F1
0,Initial development,500,100,0.0936,0.4945,0.50,0.0000
1,Larger development,2000,500,0.2192,0.6170,0.15,0.2424
2,Independent validation,2000,500,0.1016,0.5202,0.15,0.0825


Saved: kumo_development_summary.csv
